# 1. Pengunduhan & Penggabungan Dataset VNetra

Notebook ini memisahkan tugas berat CPU (download COCO) dan tugas GPU (Merge & Training).


In [ ]:
!pip install -q albumentations
from google.colab import drive
drive.mount('/content/drive')

import os
from dotenv import load_dotenv

# ponytail: load dari satu file .env di Google Drive, tak perlu set Colab Secrets tiap run
env_path = '/content/drive/MyDrive/YOLO/vnetra.env'
load_dotenv(env_path)
if os.path.exists(env_path):
    print(f"Loaded config from {env_path}")
else:
    print(f"Bikin file {env_path} berisi ROBOFLOW_API_KEY=... dan KAGGLE_USERNAME=... dll")

# --- KONFIGURASI EKSPERIMEN (BISA DIUBAH) ---
EXPERIMENT_ID = 1

DRIVE_BASE_DIR = f'/content/drive/MyDrive/YOLO/eksperimen_{EXPERIMENT_ID}'
INPUT_DIR = f'{DRIVE_BASE_DIR}/input'
OUTPUT_DIR = f'{DRIVE_BASE_DIR}/output'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import IPython
import PIL
pil_ver = PIL.__version__
print(f"Mengunci versi Pillow ke {pil_ver} untuk mencegah crash C-extension...")
IPython.get_ipython().system(f"pip install ultralytics roboflow pyyaml Pillow=={pil_ver}")

import importlib
import site
importlib.reload(site)
importlib.invalidate_caches()

print("Environment siap!")


## 1. Unduh Dataset Kustom dari Roboflow
Dataset spesialis untuk objek navigasi tunanetra (selain kendaraan/orang yang sudah ada di COCO). Mengunduh langsung dari server Roboflow ke dalam mesin GPU Google Colab.


In [ ]:
# Mengambil API Key secara otomatis dari sistem
import os
from roboflow import Roboflow

import os
# ponytail: use env vars for secrets
roboflow_key = os.environ.get('ROBOFLOW_API_KEY')
if not roboflow_key:
    raise ValueError('API Key is missing! Set ROBOFLOW_API_KEY')
rf = Roboflow(api_key=roboflow_key)

# 2. Tactile Paving Dataset (Kelas yang diambil: straight, turn, 3way, 4way, stop)
dataset_tactile = rf.workspace("raihan-aria").project("paving-tactile-detection").version(4).download("yolov11")

# 3. Pole / Tiang Dataset (Kelas yang diambil: 'pole')
dataset_pole = rf.workspace("ghost-gsj7h").project("utility-pole-aka9k").version(3).download("yolov11")

# 4. Hanging Branch / Ranting Menggantung Dataset (Kelas yang diambil: 'hanging_branch')
dataset_branch = rf.workspace("utem").project("branch-7qne7").version(2).download("yolov11")
dataset_branch3 = rf.workspace("ahmdirfnz").project("branch").version(5).download("yolov11")

# 5. Stairs / Tangga Dataset (Kelas yang diambil: 'stairs_up', 'stairs_down')
dataset_stairs = rf.workspace("jatin-sne2e").project("stairs-zqsvn").version(2).download("yolov11")
dataset_stairs2 = rf.workspace("sovar-sfwov").project("stair-detection-large").version(1).download("yolov11")

# 6. Tree / Pohon Dataset (Kelas yang diambil: 'tree')
dataset_tree = rf.workspace("tree-nqhzs").project("tree-hmf5d").version(1).download("yolov11")

# 7. Crosswalk / Zebra Cross Dataset (Kelas yang diambil: 'crosswalk')
dataset_crosswalk = rf.workspace("wqwdas").project("crosswalk-1elwe").version(1).download("yolov11")


## 2. Penggabungan (Merging) Seluruh Dataset
Menyatukan seluruh dataset (11+ Roboflow + 1 COCO) ke dalam folder `vnetra_master_dataset` sambil merekayasa ID Kelas mereka agar berurutan (0-22) secara konsisten dan membatasi jumlah maksimal objek per kelas (Rebalancing).


In [ ]:
import os
import shutil
import yaml
import glob

master_dir = "/content/vnetra_master_dataset"
if os.path.exists(master_dir): shutil.rmtree(master_dir)
for split in ['train', 'valid', 'test']:
    os.makedirs(f"{master_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{master_dir}/{split}/labels", exist_ok=True)

master_classes = [
    "pole",
    "tactile_paving_straight", "tactile_paving_turn", 
    "tactile_paving_3way", "tactile_paving_4way", "tactile_paving_stop",
    "stairs_up", "stairs_down", "crosswalk", "tree"
]
master_class_to_id = {name: idx for idx, name in enumerate(master_classes)}

global_counter = {}
global_image_counter = {}

def merge_dataset(source_path, class_mapping):
    yaml_path = os.path.join(source_path, 'data.yaml')
    if not os.path.exists(yaml_path): return
    
    with open(yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)
        original_classes = yaml_data.get('names', [])
        if isinstance(original_classes, dict):
            original_classes = [original_classes[i] for i in range(len(original_classes))]

    for split in ['train', 'valid', 'test']:
        img_dir = f"{source_path}/{split}/images"
        lbl_dir_base = f"{source_path}/{split}/labels"
        if not os.path.exists(img_dir): continue
            
        import random
        all_images = glob.glob(f"{img_dir}/*")
        random.shuffle(all_images)
            
        for img_path in all_images:
            file_name = os.path.basename(img_path)
            lbl_name = file_name.rsplit('.', 1)[0] + '.txt'
            lbl_path = f"{lbl_dir_base}/{lbl_name}"
            if not os.path.exists(lbl_path): continue
                
            lines = open(lbl_path).readlines()
            parsed_objs = []
            
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue
                orig_id = int(parts[0])
                if orig_id >= len(original_classes): continue
                c_name = str(original_classes[orig_id]).lower()
                mapped_master = next((v for k, v in class_mapping.items() if k.lower() == c_name), None)
                if mapped_master:
                    parsed_objs.append({
                        'master_class': mapped_master,
                        'box': [float(x) for x in parts[1:5]], 
                        'line_parts': parts, 'dropped': False
                    })
            
            # 1. PIXEL SIZE FILTER
            # for obj in parsed_objs:
            #     m_class = obj['master_class']
            #     if m_class in MIN_PIXEL_SIZE_GLOBAL:
            #         xc, yc, w, h = obj['box']
            #         if max(w * 640.0, h * 640.0) < MIN_PIXEL_SIZE_GLOBAL[m_class]:
            #             obj['dropped'] = True
            
            # 2. INSTANCE COUNT LIMIT PER IMAGE (Kepadatan)
            # master_counts = {}
            # for obj in parsed_objs:
            #     if not obj['dropped']:
            #         master_counts[obj['master_class']] = master_counts.get(obj['master_class'], 0) + 1
            # if any(count > MAX_INSTANCES_GLOBAL for count in master_counts.values()):
            #     continue
                
            # 3. GLOBAL CLASS LIMITS ENFORCEMENT (SOFT-LIMIT BERBASIS IMAGE)
            has_needed_class = False
            for obj in parsed_objs:
                if not obj['dropped']:
                    m_class = obj['master_class']
                    if global_counter.get(m_class, 0) < GLOBAL_CLASS_LIMITS.get(m_class, float('inf')):
                        has_needed_class = True
                        break
            
            if not has_needed_class:
                continue

            # 4. SAVE FILE IF VALID
            temp_class_counts = {}
            valid_objects = 0
            new_labels = []
            
            for obj in parsed_objs:
                if not obj['dropped']:
                    m_class = obj['master_class']
                    temp_class_counts[m_class] = temp_class_counts.get(m_class, 0) + 1
                    valid_objects += 1
                    new_labels.append(f"{master_class_to_id[m_class]} {' '.join(obj['line_parts'][1:])}\n")

            if valid_objects > 0:
                for m_class, count in temp_class_counts.items():
                    global_counter[m_class] = global_counter.get(m_class, 0) + count
                    global_image_counter[m_class] = global_image_counter.get(m_class, 0) + 1
                    
                dest_img = f"{master_dir}/{split}/images/{file_name}"
                dest_lbl = f"{master_dir}/{split}/labels/{lbl_name}"
                shutil.copy(img_path, dest_img)
                with open(dest_lbl, 'w') as f: f.writelines(new_labels)


# =====================================================================
# --- KONFIGURASI PENGATURAN KEPADATAN & LIMIT DATASET ---
# =====================================================================

# 1. Maksimal jumlah objek (semua kelas) di dalam 1 gambar
# MAX_INSTANCES_GLOBAL = 10

# 2. Minimal ukuran objek dalam piksel (width / height) agar label terlalu kecil dibuang
# MIN_PIXEL_SIZE_GLOBAL = {
    # Kosong karena pejalan kaki/navigasi seringkali membutuhkan deteksi sekecil apapun
# }

# 3. Batas maksimum jumlah INSTANCE yang akan diambil untuk tiap kelas (Master Class)
GLOBAL_CLASS_LIMITS = {
    'pole': 2000,
    'tactile_paving_straight': 1500,
    'tactile_paving_turn': 1000,
    'tactile_paving_3way': 1000,
    'tactile_paving_4way': 1000,
    'tactile_paving_stop': 1000,
    'stairs_up': 1000,
    'stairs_down': 1000,
    'crosswalk': 1100,
    'tree': 1000
}
# =====================================================================


print("Memproses Tactile Paving Dataset...")
merge_dataset(dataset_tactile.location, {"go": "tactile_paving_straight", "1": "tactile_paving_straight", "0": "tactile_paving_straight", "straight": "tactile_paving_straight", "2": "tactile_paving_turn", "3": "tactile_paving_3way", "4": "tactile_paving_4way", "stop": "tactile_paving_stop"})

print("Memproses Pole Dataset...")
merge_dataset(dataset_pole.location, {"pole": "pole", "pole_including_insulator": "pole"})

print("Memproses Stairs Dataset...")
merge_dataset(dataset_stairs.location, {"downstair": "stairs_down", "upstair": "stairs_up", "stairs_up": "stairs_up", "stairs_down": "stairs_down"})
merge_dataset(dataset_stairs2.location, {"downstair": "stairs_down", "upstair": "stairs_up"})

print("Memproses Tree Dataset...")
merge_dataset(dataset_tree.location, {"tree": "tree"})

def merge_dynamic(dataset_loc, target_class):
    try:
        with open(f"{dataset_loc}/data.yaml", 'r') as f:
            classes = yaml.safe_load(f)['names']
        if isinstance(classes, dict): classes = [classes[i] for i in range(len(classes))]
        cmap = {str(c): target_class for c in classes if str(c).lower() != "null"}
        merge_dataset(dataset_loc, cmap)
    except Exception as e:
        pass

print("Memproses Crosswalk Dataset...")
merge_dynamic(dataset_crosswalk.location, "crosswalk")

yaml_content = {
    "path": master_dir,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(master_classes),
    "names": master_classes
}
with open(f"{master_dir}/data.yaml", 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print("\n" + "="*50)
print("✅ Merge selesai! Statistik Gambar & Instance per kelas:")
for cls in master_classes:
    count = global_counter.get(cls, 0)
    img_count = global_image_counter.get(cls, 0)
    print(f"  {cls:20s}: {img_count:>6,} gambar  |  {count:>6,} instance")
print("="*50)


### 2.1 Jaring Pengaman Rebalancing Data
Mendistribusikan secara adil jumlah gambar (15%) ke dalam keranjang Validation dan Test set, sambil mengembalikan sisa gambar berlebih kembali ke Train set.

In [ ]:
import os
import random
import shutil

print("=== MEMASTIKAN DISTRIBUSI HYBRID VALIDATION & TEST SET (REBALANCING) ===")
train_img_dir = f'{master_dir}/train/images'
train_lbl_dir = f'{master_dir}/train/labels'
valid_img_dir = f'{master_dir}/valid/images'
valid_lbl_dir = f'{master_dir}/valid/labels'
test_img_dir  = f'{master_dir}/test/images'
test_lbl_dir  = f'{master_dir}/test/labels'

for dir_path in [valid_img_dir, valid_lbl_dir, test_img_dir, test_lbl_dir]:
    os.makedirs(dir_path, exist_ok=True)

def get_class_counts(lbl_dir):
    counts = {i: 0 for i in range(len(master_classes))}
    if not os.path.exists(lbl_dir): return counts
    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(lbl_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    if cls_id in counts: counts[cls_id] += 1
    return counts

train_counts = get_class_counts(train_lbl_dir)
valid_counts = get_class_counts(valid_lbl_dir)
test_counts  = get_class_counts(test_lbl_dir)

total_counts = {}
target_valid_test = {}

for cls_id in range(len(master_classes)):
    total = train_counts[cls_id] + valid_counts[cls_id] + test_counts[cls_id]
    total_counts[cls_id] = total
    if total > 0:
        # Ambil 15% dari total, dengan batas maksimum 500 dan minimum 1
        target = max(1, int(0.15 * total))
        target_valid_test[cls_id] = min(target, 500)
    else:
        target_valid_test[cls_id] = 0

def balance_split(target_split_name, target_img_dir, target_lbl_dir, current_counts):
    classes_to_boost = [c for c in range(len(master_classes)) if current_counts[c] < target_valid_test[c]]
    
    if not classes_to_boost:
        print(f"Semua kelas sudah mencapai target hybrid di {target_split_name} Set! Aman.")
        return current_counts

    print(f"Ada kelas yang kurang data di {target_split_name} Set: {classes_to_boost}")
    print(f"Meminjam gambar secara acak dari folder Train untuk {target_split_name}...")
    
    train_labels = [f for f in os.listdir(train_lbl_dir) if f.endswith('.txt')]
    random.shuffle(train_labels)
    
    moved_images = 0
    for lbl_file in train_labels:
        if not classes_to_boost: break
        
        src_lbl = os.path.join(train_lbl_dir, lbl_file)
        lines = open(src_lbl).readlines()
        classes_in_file = {int(line.split()[0]) for line in lines if line.strip()}
        contains_needed_class = any(c in classes_to_boost for c in classes_in_file)
                
        if contains_needed_class:
            dst_lbl = os.path.join(target_lbl_dir, lbl_file)
            img_file_base = os.path.splitext(lbl_file)[0]
            
            src_img, dst_img = None, None
            for ext in ['.jpg', '.jpeg', '.png']:
                temp_src = os.path.join(train_img_dir, img_file_base + ext)
                if os.path.exists(temp_src):
                    src_img = temp_src
                    dst_img = os.path.join(target_img_dir, img_file_base + ext)
                    break
            
            if src_img and os.path.exists(src_img):
                shutil.move(src_img, dst_img)
                shutil.move(src_lbl, dst_lbl)
                moved_images += 1
                
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        c_id = int(parts[0])
                        if c_id in current_counts: 
                            current_counts[c_id] += 1
                            
                classes_to_boost = [c for c in range(len(master_classes)) if current_counts[c] < target_valid_test[c]]

    print(f"Berhasil memindahkan {moved_images} gambar dari Train ke {target_split_name}!")
    return current_counts

print("\n--- 1. HYBRID REBALANCING VALIDATION SET ---")
valid_counts = balance_split("Validation", valid_img_dir, valid_lbl_dir, valid_counts)

print("\n--- 2. HYBRID REBALANCING TEST SET ---")
test_counts = balance_split("Test", test_img_dir, test_lbl_dir, test_counts)

print("\\n=== MENGEMBALIKAN KELEBIHAN GAMBAR KE FOLDER TRAIN (STRICT CAPPING) ===")
train_counts = get_class_counts(train_lbl_dir)

def return_excess_to_train(source_name, source_img_dir, source_lbl_dir, current_counts):
    # DIBUANG: and train_counts[c] < current_counts[c]
    classes_to_reduce = [c for c in range(len(master_classes)) if current_counts[c] > target_valid_test[c]]
    
    if not classes_to_reduce:
        return current_counts
        
    print(f"Mengembalikan kelebihan data dari {source_name} ke Train untuk kelas: {classes_to_reduce}")
    
    labels_list = [f for f in os.listdir(source_lbl_dir) if f.endswith('.txt')]
    random.shuffle(labels_list)
    moved_back = 0
    
    for lbl_file in labels_list:
        if not classes_to_reduce: break
        
        src_lbl = os.path.join(source_lbl_dir, lbl_file)
        lines = open(src_lbl).readlines()
        classes_in_file = {int(line.split()[0]) for line in lines if line.strip()}
        contains_excess_class = any(c in classes_to_reduce for c in classes_in_file)
                
        if contains_excess_class:
            dst_lbl = os.path.join(train_lbl_dir, lbl_file)
            img_file_base = os.path.splitext(lbl_file)[0]
            
            src_img, dst_img = None, None
            for ext in ['.jpg', '.jpeg', '.png']:
                temp_src = os.path.join(source_img_dir, img_file_base + ext)
                if os.path.exists(temp_src):
                    src_img = temp_src
                    dst_img = os.path.join(train_img_dir, img_file_base + ext)
                    break
            
            if src_img and os.path.exists(src_img):
                can_move = True
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        c_id = int(parts[0])
                        if current_counts[c_id] <= target_valid_test[c_id]:
                            can_move = False
                            break
                
                if can_move:
                    shutil.move(src_img, dst_img)
                    shutil.move(src_lbl, dst_lbl)
                    moved_back += 1
                    
                    for line in lines:
                        parts = line.strip().split()
                        if parts:
                            c_id = int(parts[0])
                            current_counts[c_id] -= 1
                            train_counts[c_id] += 1
                                
                    classes_to_reduce = [c for c in range(len(master_classes)) if current_counts[c] > target_valid_test[c]]
                    
    print(f"Berhasil mengembalikan {moved_back} gambar dari {source_name} ke Train!")
    return current_counts

valid_counts = return_excess_to_train("Validation", valid_img_dir, valid_lbl_dir, valid_counts)
test_counts = return_excess_to_train("Test", test_img_dir, test_lbl_dir, test_counts)
print("\nDistribusi Hybrid Selesai! Model akan aman dari Catastrophic Forgetting untuk kelas kecil.")



### 2.2 Laporan Akhir Proporsi Dataset
Mencetak tabel distribusi dari keseluruhan dataset untuk verifikasi manual.

In [ ]:
import pandas as pd
import os

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images(f'{master_dir}/train/images')
valid_count = count_images(f'{master_dir}/valid/images')
test_count  = count_images(f'{master_dir}/test/images')
total_images = train_count + valid_count + test_count

print("=== Statistik Keseluruhan ===")
print(f"Total Lembar Gambar (All) : {total_images} gambar")
print(f"Total Gambar Training     : {train_count} gambar")
print(f"Total Gambar Validasi     : {valid_count} gambar")
print(f"Total Gambar Testing      : {test_count} gambar")
print("=============================")
print("")

def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts: counts[int(parts[0])] += 1
    return counts

train_cls = count_instances_per_class(f'{master_dir}/train/labels', len(master_classes))
valid_cls = count_instances_per_class(f'{master_dir}/valid/labels', len(master_classes))
test_cls  = count_instances_per_class(f'{master_dir}/test/labels', len(master_classes))

data_report = []
total_train = 0
total_valid = 0
total_test = 0
global_total = 0

for i, cls_name in enumerate(master_classes):
    t_train = train_cls[i]
    t_valid = valid_cls[i]
    t_test = test_cls[i]
    t_total = t_train + t_valid + t_test
    
    total_train += t_train
    total_valid += t_valid
    total_test += t_test
    global_total += t_total
    
    data_report.append({
        'ID': i, 
        'Kelas': cls_name, 
        'Train (Inst)': t_train, 
        'Valid (Inst)': t_valid, 
        'Test (Inst)': t_test,
        'Total Instance': t_total
    })

data_report.append({
    'ID': '-', 
    'Kelas': 'TOTAL KESELURUHAN', 
    'Train (Inst)': total_train, 
    'Valid (Inst)': total_valid, 
    'Test (Inst)': total_test,
    'Total Instance': global_total
})

df_report = pd.DataFrame(data_report)
display(df_report)


### 2.3 Backup Dataset ke Google Drive (Wajib untuk Resume)
Dataset yang sudah digabung akan di-ZIP dan dikirim langsung ke `INPUT_DIR` di Google Drive Anda agar notebook *Resume* dapat mengambilnya kembali jika sesi Colab ini terputus.

In [ ]:
import shutil
import os

print("📦 Membuat arsip ZIP untuk seluruh dataset master...")
dataset_dir = master_dir  
zip_path = f'{INPUT_DIR}/vnetra_master_dataset'

try:
    os.makedirs(INPUT_DIR, exist_ok=True)
    shutil.make_archive(zip_path, 'zip', dataset_dir)
    print(f"✅ Selesai! Dataset master telah diamankan secara permanen ke: {zip_path}.zip")
    print("Di masa depan, Anda bisa menggunakan notebook Resume untuk memanggil dataset ini!")
except Exception as e:
    print(f"❌ Gagal melakukan backup: {e}")
    print("Pastikan Anda telah mengizinkan Google Colab untuk mengakses Google Drive Anda di cell paling atas.")
